# Notebook 04 — Causal masking and KV-cache

Paired with `theory/04_causal_masking.md`. **Mode:** theory-first.

## QHMPC

**Question.** Verify five structural properties of causal attention and its inference-time optimization:
1. Additive $-\infty$ masking produces lower-triangular attention matrices to fp precision.
2. Multiplicative-mask-after-softmax is a bug: rows do not sum to one.
3. KV-cache inference produces identical outputs to full-matrix causal attention.
4. FLOP counts: training ($O(n^2)$) vs. KV-cached inference ($\sum_t O(t) = O(n^2)$) match asymptotically.
5. On a toy sequence, visualize how causal attention at position $t$ distributes mass over $\{1, \dots, t\}$.

**Hypothesis.** Props 4.2 and 4.3 are exact. The experiments confirm and put numbers on FLOPs / memory. The interesting output is in Experiment 5 — attention patterns on a hand-crafted sequence.

**Method.** NumPy/torch, no training, laptop CPU.

**Prediction.** *Paste §4.7 predictions.*

1. Mask correctness — 
2. Multiplicative-mask bug — 
3. KV-cache equivalence max-abs error — 
4. FLOP ratio training vs. KV-cached inference at n=1024 — 
5. Toy-sequence attention row predictions — 

**Confidence.** Overall: *[low / medium / high, reason]*.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)
np.set_printoptions(precision=4, suppress=True)

---
## Experiment 1 — causal mask produces lower-triangular attention

Additive $-\infty$ mask on the upper triangle, then row-wise softmax.

In [ ]:
def causal_attention(Q, K, V):
    d_k = Q.shape[-1]
    n, m = Q.shape[0], K.shape[0]
    scores = Q @ K.T / np.sqrt(d_k)
    mask = np.triu(np.full((n, m), -np.inf), k=1)
    scores = scores + mask
    scores = scores - np.nanmax(scores, axis=-1, keepdims=True)
    e = np.exp(scores)
    e[np.isnan(e)] = 0.0       # -inf - -inf = nan in shift
    w = e / e.sum(axis=-1, keepdims=True)
    return w @ V, w

n, d_k, d_v = 6, 8, 4
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_v)

out, w = causal_attention(Q, K, V)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(w, cmap='viridis', vmin=0, vmax=w.max())
for i in range(n):
    for j in range(n):
        if w[i, j] > 0:
            ax.text(j, i, f'{w[i, j]:.2f}', ha='center', va='center',
                    color='white' if w[i, j] < w.max()/2 else 'black', fontsize=8)
ax.set_xlabel('key (source)'); ax.set_ylabel('query'); ax.set_title('Causal attention matrix')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

print(f'Max upper-triangular entry (should be 0): {w[np.triu_indices(n, k=1)].max():.2e}')
print(f'Row sums (should all be 1): {w.sum(axis=1)}')

**Sub-finding 1.** *Lower triangular confirmed? Row sums exactly 1?*

---
## Experiment 2 — multiplicative-mask-after-softmax is a bug

Wrong implementation: softmax first, then zero out the upper triangle. Rows will not sum to 1.

In [ ]:
def buggy_mask_after_softmax(Q, K, V):
    d_k = Q.shape[-1]
    n = Q.shape[0]
    scores = Q @ K.T / np.sqrt(d_k)
    scores = scores - scores.max(axis=-1, keepdims=True)
    e = np.exp(scores); w = e / e.sum(axis=-1, keepdims=True)  # full (no mask yet)
    mask = np.tril(np.ones((n, n)))
    w_bug = w * mask                                            # zero out upper
    return w_bug @ V, w_bug

_, w_bug = buggy_mask_after_softmax(Q, K, V)
print(f'Buggy attention row sums (should NOT all be 1):')
print(w_bug.sum(axis=1))
print(f'\nMissing mass per row: {1 - w_bug.sum(axis=1)}')
print(f'Exactly equals what would have gone to the future: {(1 - w_bug.sum(axis=1))}')

**Sub-finding 2.** *How much mass "leaks" on early rows? What would a naive renormalization-fix do differently from the correct additive-mask version? (Hint: the Jacobian differs.)*

---
## Experiment 3 — KV-cache produces identical output (Prop 4.3)

Generate the same $n$ outputs two ways:
- (A) full-matrix causal attention on all $n$ positions.
- (B) sequential KV-cache: at step $t$, attend from $q_t$ to $K_{1:t}, V_{1:t}$.

In [ ]:
n, d_k, d_v = 16, 32, 32
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_v)

# (A) full-matrix
out_A, _ = causal_attention(Q, K, V)

# (B) sequential with KV-cache
out_B = np.zeros((n, d_v))
for t in range(n):
    Qt = Q[t:t+1]                                 # (1, d_k)
    Kt = K[:t+1]                                  # (t+1, d_k) — cached up to t
    Vt = V[:t+1]                                  # (t+1, d_v)
    scores = Qt @ Kt.T / np.sqrt(d_k)
    scores = scores - scores.max()
    w = np.exp(scores); w = w / w.sum()
    out_B[t] = (w @ Vt).squeeze()

print(f'Max-abs difference A vs B: {np.max(np.abs(out_A - out_B)):.2e}')
print(f'(should be at floating-point precision)')

**Sub-finding 3.** *Equivalence to fp precision? Any systematic drift from the shift/normalizer computation?*

---
## Experiment 4 — FLOP count: training vs. KV-cached inference

Count the dominant FLOPs (matrix-multiply ops, 2 FLOPs per multiply-add):
- Training pass: one $QK^\top$ of shape $(n, d_k) @ (d_k, n) \to (n, n)$: $2 n^2 d_k$ FLOPs. Plus the $A V$ product: $2 n^2 d_v$.
- KV-cache generation: sum over $t = 1, \dots, n$ of $\mathbf{q}_t K_{1:t}^\top$: $2 t d_k$ FLOPs per step, total $n(n+1) d_k \approx n^2 d_k$. Plus $A V$: total $\approx n^2 d_v$.

In [ ]:
ns = [8, 32, 128, 512, 1024]
d_k = 64

print(f'{"n":>6}  {"train FLOPs":>16}  {"KV-cache FLOPs":>18}  {"ratio":>8}')
print('-' * 54)
for n in ns:
    train = 2 * n * n * d_k       # QK^T only (drop AV since same)
    kvc = 2 * sum(t * d_k for t in range(1, n + 1))
    print(f'{n:>6}  {train:>16}  {kvc:>18}  {kvc/train:>8.3f}')

# Asymptotic: KV-cache / train = n(n+1)/(2 n^2) -> 1/2 as n -> inf
print('\nAsymptotic ratio: KV-cache/train -> 1/2 (since sum_{t=1}^n t = n(n+1)/2 vs. n^2)')

**Sub-finding 4.** *Did the ratio converge to 1/2? Why half? (Each KV-cache step attends only to the past, not the full sequence — averaging over steps gives a factor of ~1/2.)*

---
## Experiment 5 — attention patterns on a toy causal sequence

Five-token sequence with hand-crafted embeddings. Visualize the causal attention matrix and check which positions attract mass from later queries.

In [ ]:
rng = np.random.default_rng(0)
d = 16
# Hand-crafted embeddings: token 2 shares signal with tokens 0 and 4; token 3 is neutral; token 1 is orthogonal.
e_A = rng.normal(size=d); e_B = rng.normal(size=d); e_C = rng.normal(size=d)
X = np.stack([
    e_A + 0.05 * rng.normal(size=d),
    e_B + 0.05 * rng.normal(size=d),
    0.5 * e_A + 0.5 * e_C,
    e_C + 0.05 * rng.normal(size=d),
    e_A + 0.05 * rng.normal(size=d),
])
_, W = causal_attention(X, X, X)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(W, cmap='viridis', vmin=0)
for i in range(5):
    for j in range(5):
        if W[i, j] > 0:
            ax.text(j, i, f'{W[i, j]:.2f}', ha='center', va='center',
                    color='white' if W[i, j] < W.max()/2 else 'black', fontsize=9)
ax.set_xlabel('key position'); ax.set_ylabel('query position')
ax.set_title('Causal self-attention (Q=K=V=X)')
ax.set_xticks(range(5)); ax.set_yticks(range(5))
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

# The interesting row: query at position 4 (last token, signal A) — how does it split mass?
print(f'Row 4 (query = signal A): {W[4]}')

**Sub-finding 5.** *Where did row 4 concentrate mass — on position 0 (early A), position 2 (half-A), or itself? Does the causal constraint force any interesting structure?*

---
## Finding

*3–6 sentences. What held, what missed, what surprised. What does this say about the training/inference symmetry — is it exact, or are there subtle places (numerical precision, fp16 softmax) where the two diverge in production?*